In [1]:
import requests
import os
import time
import sys
import json
sys.path.append("../Assets")
from Assets.Cities import Cities
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
api_key1 = os.getenv("OPENWEATHER_API_KEY")
api_key2 = os.getenv("WAQIP_API_KEY")
print(f"API Key loaded: {api_key1[:5]}..." if api_key1 else "ERROR: Key not found")
print(f"API Key loaded: {api_key2[:5]}..." if api_key2 else "ERROR: Key not found")

API Key loaded: a71d7...
API Key loaded: 05947...


In [3]:
def get_weather(city_name):
    """
    Fetch weather data for a city with error handling.

    Args:
        city_name (str): Name of the city

    Returns:
        dict: Weather data if successful, None if failed
    """
    # Build the API URL
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={api_key1}&units=metric"

    try:
        # Send request with 10-second timeout
        response = requests.get(url, timeout=10)

        # Check status code and return appropriate result
        if response.status_code == 200:
            return response.json()  # Success: return data
        elif response.status_code == 404:
            print(f"  ✗ Error: City '{city_name}' not found")
            return None
        elif response.status_code == 401:
            print(f"  ✗ Error: Invalid API key")
            return None
        else:
            print(f"  ✗ Error: API returned status {response.status_code}")
            return None

    except requests.exceptions.Timeout:
        print(f"  ✗ Error: Request timed out for '{city_name}'")
        return None
    except requests.exceptions.RequestException as e:
        print(f"  ✗ Network error: {e}")
        return None

In [4]:
def get_air_quality(city_name):
    """
    Fetch air quality data from WAQI.

    Returns:
        dict: Clean air quality data or None if failed
    """
    url = f"https://api.waqi.info/feed/{city_name}/?token={api_key2}"

    try:
        response = requests.get(url, timeout=10)
        data = response.json()

        if data.get("status") == "ok":
            aqi_data = data["data"]
            return {
                'city': city_name,
                'aqi': aqi_data.get('aqi'),
                'pm25': aqi_data['iaqi'].get('pm25', {}).get('v') if 'iaqi' in aqi_data else None,
                'pm10': aqi_data['iaqi'].get('pm10', {}).get('v') if 'iaqi' in aqi_data else None,
                'o3': aqi_data['iaqi'].get('o3', {}).get('v') if 'iaqi' in aqi_data else None,
                'no2': aqi_data['iaqi'].get('no2', {}).get('v') if 'iaqi' in aqi_data else None,
                'so2': aqi_data['iaqi'].get('so2', {}).get('v') if 'iaqi' in aqi_data else None,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }
        else:
            print(f"  ✗ Air Quality: {data.get('data', 'Unknown error')}")
            return None

    except Exception as e:
        print(f"  ✗ Air Quality: {str(e)[:50]}")
        return None


In [5]:
def fetch_all_data(cities_list, delay=1.0):
    """
    Fetch weather and air quality for all cities.

    Args:
        cities_list: List of city names
        delay: Seconds between API calls (default 1.0)

    Returns:
        tuple: (weather_data_list, air_quality_data_list)
    """
    weather_results = []
    air_quality_results = []

    print(f"Fetching data for {len(cities_list)} cities...\n")
    start_time = time.time()

    for i, city in enumerate(cities_list, 1):
        print(f"[{i}/{len(cities_list)}] {city}")

        # Fetch weather
        weather = get_weather(city)
        if weather:
            weather_results.append(weather)
            print(f"  ✓ Weather: {weather['temperature']}°C, {weather['description']}")
        else:
            print(f"  ✗ Weather: Failed")

        # Fetch air quality
        air = get_air_quality(city)
        if air:
            air_quality_results.append(air)
            print(f"  ✓ Air Quality: AQI {air['aqi']}")
        else:
            print(f"  ✗ Air Quality: Failed")

        # Delay between cities (except last one)
        if i < len(cities_list):
            time.sleep(delay)

    duration = time.time() - start_time
    print(f"\n{'='*50}")
    print(f"Done in {duration:.1f} seconds")
    print(f"Weather: {len(weather_results)}/{len(cities_list)} cities")
    print(f"Air Quality: {len(air_quality_results)}/{len(cities_list)} cities")

    return weather_results, air_quality_results


In [6]:
get_air_quality("Delhi")

{'city': 'Delhi',
 'aqi': 168,
 'pm25': 168,
 'pm10': 164,
 'o3': 6.8,
 'no2': 34.9,
 'so2': 18,
 'timestamp': '2026-04-20 09:56:54'}

In [10]:
city = "Delhi"
url = f"https://api.waqi.info/feed/{city}/?token={api_key2}"

response = requests.get(url, timeout=10)

data = response.json()

data

{'status': 'ok',
 'data': {'aqi': 295,
  'idx': 10111,
  'attributions': [{'url': 'http://dpccairdata.com/',
    'name': 'Delhi Pollution Control Commitee (Government of NCT of Delhi)',
    'logo': 'India-DPCCC.png'},
   {'url': 'https://waqi.info/', 'name': 'World Air Quality Index Project'}],
  'city': {'geo': [28.612498, 77.237388],
   'name': 'Major Dhyan Chand National Stadium, Delhi, Delhi, India',
   'url': 'https://aqicn.org/city/delhi/major-dhyan-chand-national-stadium',
   'location': ''},
  'dominentpol': 'pm10',
  'iaqi': {'co': {'v': 7.3},
   'dew': {'v': 14.5},
   'h': {'v': 35.2},
   'no2': {'v': 23.1},
   'o3': {'v': 18.8},
   'p': {'v': 985},
   'pm10': {'v': 295},
   'pm25': {'v': 187},
   'so2': {'v': 18.6},
   't': {'v': 31.3},
   'w': {'v': 1.1},
   'wd': {'v': 84.5},
   'wg': {'v': 16.7}},
  'time': {'s': '2026-04-20 10:00:00',
   'tz': '+05:30',
   'v': 1776679200,
   'iso': '2026-04-20T10:00:00+05:30'},
  'forecast': {'daily': {'pm10': [{'avg': 46,
      'day': 

NameError: name 'response' is not defined